In [1]:
"""
SANITY TEST PIPELINE
Generate one env → generate pairwise + improvement → verify correctness
"""

import os
import sys
import numpy as np

# ------------------------------------------------------------------
# Path setup (same as experiment pipeline)
# ------------------------------------------------------------------
module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# ------------------------------------------------------------------
# Imports (same modules as experiment pipeline)
# ------------------------------------------------------------------
from utils import (
    generate_random_gridworld_envs,
    parallel_value_iteration,
    GenerationSpec,
    DemoSpec,
    FeedbackSpec,
)

from utils.feedback_budgeting import generate_candidate_atoms_for_scot
from gridworld_env_layout import GridWorldMDPFromLayoutEnv

# ------------------------------------------------------------------
# Helper: trajectory reward (same logic used in generation)
# ------------------------------------------------------------------

def trajectory_return(env, traj, gamma=1.0):
    total = 0.0
    for t, (s, a) in enumerate(traj):
        if a is None:
            break
        r = env.compute_reward(s)
        total += (gamma ** t) * r
    return float(total)

# ------------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------------

def check_pairwise(atoms_per_env, envs):

    total = 0
    errors = 0

    for env_atoms in atoms_per_env:
        for atom in env_atoms:

            if atom.feedback_type != "pairwise":
                continue

            env = envs[atom.env_idx]
            pref, other = atom.data

            r_pref = trajectory_return(env, pref)
            r_other = trajectory_return(env, other)

            total += 1

            if r_pref <= r_other:
                errors += 1
                print("PAIRWISE ERROR")
                print("env:", atom.env_idx)
                print("preferred:", r_pref)
                print("other:", r_other)
                print()

    print("Pairwise checked:", total)
    print("Pairwise errors:", errors)


def check_improvement(atoms_per_env, envs):

    total = 0
    errors = 0

    for env_atoms in atoms_per_env:
        for atom in env_atoms:

            if atom.feedback_type != "improvement":
                continue

            env = envs[atom.env_idx]
            improved, original = atom.data

            r_imp = trajectory_return(env, improved)
            r_orig = trajectory_return(env, original)

            total += 1

            if r_imp <= r_orig:
                errors += 1
                print("IMPROVEMENT ERROR")
                print("env:", atom.env_idx)
                print("improved:", r_imp)
                print("original:", r_orig)
                print()

    print("Improvement checked:", total)
    print("Improvement errors:", errors)

# ------------------------------------------------------------------
# 1. Generate ONE environment
# ------------------------------------------------------------------

seed = 0
feature_dim = 2
grid_size = 10

rng = np.random.default_rng(seed)

# ground truth reward
w_true = rng.normal(size=feature_dim)
w_true = w_true / np.linalg.norm(w_true)

color_to_feature_map = {
    f"f{i}": [1 if j == i else 0 for j in range(feature_dim)]
    for i in range(feature_dim)
}

envs, _ = generate_random_gridworld_envs(
    n_envs=1,
    rows=grid_size,
    cols=grid_size,
    color_to_feature_map=color_to_feature_map,
    palette=list(color_to_feature_map.keys()),
    p_color_range={c: (0.3, 0.8) for c in color_to_feature_map},
    terminal_policy=dict(kind="random_k", k_min=1, k_max=1),
    gamma_range=(0.99, 0.99),
    noise_prob_range=(0.0, 0.0),
    w_mode="fixed",
    W_fixed=w_true,
    seed=seed,
    GridEnvClass=GridWorldMDPFromLayoutEnv,
)

print("Generated environment")

# ------------------------------------------------------------------
# 2. Compute optimal Q
# ------------------------------------------------------------------

Q_list = parallel_value_iteration(envs, epsilon=1e-10)

print("Computed Q")

# ------------------------------------------------------------------
# 3. Generate feedback atoms (pairwise + improvement only)
# ------------------------------------------------------------------

spec = GenerationSpec(
    seed=seed,

    demo=DemoSpec(enabled=False),

    pairwise=FeedbackSpec(
        enabled=True,
        total_budget=50,
        alloc_method="uniform",
    ),

    estop=FeedbackSpec(
        enabled=False,
        total_budget=0,
    ),

    improvement=FeedbackSpec(
        enabled=True,
        total_budget=50,
        alloc_method="uniform",
    ),
)

atoms_per_env = generate_candidate_atoms_for_scot(
    envs,
    Q_list,
    spec=spec
)

print("Generated atoms")

# ------------------------------------------------------------------
# 4. Count atoms
# ------------------------------------------------------------------

pairwise_count = 0
improvement_count = 0

for atoms in atoms_per_env:
    for a in atoms:
        if a.feedback_type == "pairwise":
            pairwise_count += 1
        if a.feedback_type == "improvement":
            improvement_count += 1

print("Pairwise atoms:", pairwise_count)
print("Improvement atoms:", improvement_count)

# ------------------------------------------------------------------
# 5. Run sanity checks
# ------------------------------------------------------------------

print("\nRunning pairwise check")
check_pairwise(atoms_per_env, envs)

print("\nRunning improvement check")
check_improvement(atoms_per_env, envs)

print("\nSANITY TEST FINISHED")

Generated environment
[3/12] Running Value Iteration on all MDPs... (parallel)
       VI progress: 1/1 MDPs solved...
       ✔ VI completed in 1.35s

Computed Q
Generated atoms
Pairwise atoms: 50
Improvement atoms: 50

Running pairwise check
Pairwise checked: 50
Pairwise errors: 0

Running improvement check
Improvement checked: 50
Improvement errors: 0

SANITY TEST FINISHED
